# 05 多工具调用与Pydantic

**用途：** 运行确定性workflow，观察意图、槽位、工具参数、结果和Pydantic失败。

> 使用方式：按顺序运行。出现 `PASS` 才代表本节验收成功；断言失败时先阅读紧邻的“失败定位”。默认不调用真实模型、不写生产数据库。

In [1]:
from pathlib import Path
import importlib.util
import json
import os
import sys
import tempfile

cwd = Path.cwd().resolve()
DAY1_ROOT = None
PROJECT2_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "project2" / "agent_graph.py").exists():
        DAY1_ROOT = candidate
        PROJECT2_ROOT = candidate / "project2"
        break
    if (candidate / "agent_graph.py").exists() and (candidate / "tests").exists():
        PROJECT2_ROOT = candidate
        DAY1_ROOT = candidate.parent
        break
assert DAY1_ROOT is not None and PROJECT2_ROOT is not None, "找不到 day1/project2 项目根目录"
NOTEBOOK_ROOT = PROJECT2_ROOT / "notebooks"
for path in [str(DAY1_ROOT), str(PROJECT2_ROOT), str(NOTEBOOK_ROOT)]:
    if path not in sys.path:
        sys.path.insert(0, path)

from notebook_utils import (
    check,
    check_equal,
    file_inventory,
    load_jsonl,
    masked_environment,
    run_command,
    run_unittest,
    show_markdown,
    show_table,
    source_excerpt,
)

RUN_LIVE_MODEL_TESTS = os.getenv("RUN_LIVE_MODEL_TESTS", "0") == "1"
print(f"Python: {sys.executable}")
print(f"DAY1_ROOT: {DAY1_ROOT}")
print(f"PROJECT2_ROOT: {PROJECT2_ROOT}")
print(f"RUN_LIVE_MODEL_TESTS: {RUN_LIVE_MODEL_TESTS}")

Python: D:\new things\项目1\day1\.venv\Scripts\python.exe
DAY1_ROOT: D:\new things\项目1\day1
PROJECT2_ROOT: D:\new things\项目1\day1\project2
RUN_LIVE_MODEL_TESTS: False


In [2]:
from agent_workflow import run_agent

question = "小松PC200原厂液压泵要1件，有没有现货，多少钱，发到贵阳要多久？"
result = run_agent(question)
show_table([{
    "状态": result["status"],
    "意图": "、".join(result["parse_result"]["intents"]),
    "工具": "、".join(result["called_tools"]),
    "缺失字段": "、".join(result["parse_result"]["missing_fields"]),
}])
print(json.dumps(result["tool_arguments"], ensure_ascii=False, indent=2))
print(json.dumps(result["tool_results"], ensure_ascii=False, indent=2))
check_equal(
    "按意图调用三种工具",
    result["called_tools"],
    ["inventory_tool", "quote_tool", "logistics_tool"],
)

,状态,意图,工具,缺失字段
0,completed,inventory、quote、logistics,inventory_tool、quote_tool、logistics_tool,


{
  "inventory_tool": {
    "brand": "小松",
    "machine_model": "PC200",
    "part_name": "液压泵",
    "quality_level": "原厂"
  },
  "quote_tool": {
    "brand": "小松",
    "machine_model": "PC200",
    "part_name": "液压泵",
    "quality_level": "原厂",
    "quantity": 1
  },
  "logistics_tool": {
    "city": "贵阳",
    "part_name": "液压泵",
    "urgent": null
  }
}
{
  "inventory_tool": {
    "matched": true,
    "in_stock": true,
    "stock_count": 2,
    "warehouse": "贵阳仓",
    "need_manual_confirm": true,
    "message": "查询到模拟库存，正式库存仍需人工确认。",
    "options": [
      {
        "part_id": "P001",
        "brand": "小松",
        "machine_model": "PC200",
        "part_name": "液压泵",
        "quality_level": "原厂",
        "stock_count": 2,
        "warehouse": "贵阳仓",
        "status": "available"
      }
    ]
  },
  "quote_tool": {
    "matched": true,
    "part_id": "P001",
    "brand": "小松",
    "machine_model": "PC200",
    "part_name": "液压泵",
    "quality_level": "原厂",
    "quantity": 1,
    "u

{'检查项': '按意图调用三种工具',
 '状态': 'PASS',
 '说明': "actual=['inventory_tool', 'quote_tool', 'logistics_tool'], expected=['inventory_tool', 'quote_tool', 'logistics_tool']"}

In [3]:
from pydantic import ValidationError
from schemas import QuoteToolArgs

validation_failed = False
try:
    QuoteToolArgs(
        machine_model="PC200",
        part_name="液压泵",
        quality_level="原厂",
        quantity=0,
    )
except ValidationError as exc:
    validation_failed = True
    print(exc)
check("非法数量在调用工具前被拒绝", validation_failed)

1 validation error for QuoteToolArgs
quantity
  Input should be greater than or equal to 1 [type=greater_than_equal, input_value=0, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than_equal
[PASS] 非法数量在调用工具前被拒绝


{'检查项': '非法数量在调用工具前被拒绝', '状态': 'PASS', '说明': ''}

In [4]:
source_excerpt(PROJECT2_ROOT / "tool_dispatcher.py", 1, 115)

   1: from __future__ import annotations
   2: 
   3: from typing import Any
   4: 
   5: from langchain_tools import get_langchain_tool_map
   6: from schemas import (
   7:     InventoryToolArgs,
   8:     KnowledgeToolArgs,
   9:     LogisticsToolArgs,
  10:     QuoteToolArgs,
  11:     TicketToolArgs,
  12:     dump_args,
  13: )
  14: 
  15: 
  16: TOOL_REGISTRY = get_langchain_tool_map()
  17: SUPPORTED_TOOLS = set(TOOL_REGISTRY)
  18: TOOL_ARG_MODELS = {
  19:     tool_name: tool.args_schema
  20:     for tool_name, tool in TOOL_REGISTRY.items()
  21: }
  22: 
  23: 
  24: def build_tool_args(tool_name: str, parse_result: dict[str, Any]) -> dict[str, Any]:
  25:     slots = parse_result["slots"]
  26: 
  27:     if tool_name == "inventory_tool":
  28:         return dump_args(
  29:             InventoryToolArgs(
  30:                 brand=slots.get("brand"),
  31:                 machine_model=slots["machine_model"],
  32:                 part_name=slots["part_name"],
  33:   

'   1: from __future__ import annotations\n   2: \n   3: from typing import Any\n   4: \n   5: from langchain_tools import get_langchain_tool_map\n   6: from schemas import (\n   7:     InventoryToolArgs,\n   8:     KnowledgeToolArgs,\n   9:     LogisticsToolArgs,\n  10:     QuoteToolArgs,\n  11:     TicketToolArgs,\n  12:     dump_args,\n  13: )\n  14: \n  15: \n  16: TOOL_REGISTRY = get_langchain_tool_map()\n  17: SUPPORTED_TOOLS = set(TOOL_REGISTRY)\n  18: TOOL_ARG_MODELS = {\n  19:     tool_name: tool.args_schema\n  20:     for tool_name, tool in TOOL_REGISTRY.items()\n  21: }\n  22: \n  23: \n  24: def build_tool_args(tool_name: str, parse_result: dict[str, Any]) -> dict[str, Any]:\n  25:     slots = parse_result["slots"]\n  26: \n  27:     if tool_name == "inventory_tool":\n  28:         return dump_args(\n  29:             InventoryToolArgs(\n  30:                 brand=slots.get("brand"),\n  31:                 machine_model=slots["machine_model"],\n  32:                 part_n

## 为什么工具必须是确定性的

LLM负责理解“客户想做什么”，库存CSV、报价规则、物流规则和工单函数负责“真实执行”。缺字段时先追问；Pydantic拒绝未知字段、空字符串和非法数量；高风险工具再进入审批。

### 面试官会问

1. 五个工具的输入输出分别是什么？
2. 为什么模型不能直接生成库存和价格？
3. Pydantic校验后为什么仍需要业务必填检查？
4. 幂等键怎样避免checkpoint恢复时重复调用？
5. 工具失败时重试、兜底和转人工的边界是什么？

### 参考答案

1. **五个工具输入输出是什么？** `inventory_tool`接收品牌、机型、配件和品质，返回匹配库存；`quote_tool`再接收数量，返回报价草稿；`logistics_tool`接收城市等信息，返回时效和运费估算；`ticket_tool`接收订单与问题，返回售后工单草稿；`knowledge_tool`接收问题，返回RAG答案、距离和来源。
2. **为什么模型不能直接生成库存和价格？** 这些值会变化且涉及业务承诺。模型只负责把自然语言转成受约束的意图和参数，真实值由确定性数据源计算，回复层也只能引用工具结果。
3. **Pydantic后为什么还要业务必填检查？** Pydantic验证类型、范围和未知字段；业务必填依赖意图和阶段，例如查物流必须有城市，售后必须有订单号，报价还要品质和数量，这种跨字段、条件式规则由workflow处理。
4. **幂等键如何工作？** 根据`request_id + tool_name + canonical arguments`生成稳定键，执行前查询记录；若checkpoint恢复时同一调用已有成功结果，就复用结果而不是再次触发外部动作。
5. **错误边界怎么划分？** 超时、连接错误等临时故障有限重试；参数和业务校验错误不重试而是追问；重试耗尽或高风险不确定结果生成解释性兜底，并在接管模式下创建人工服务单。

**代码落点：** `schemas.py`、`tool_dispatcher.py`、`tool_call_logger.py`、`agent_graph.py`和`tools/`。

**成功标准：** 三个意图只调用三个相关工具，`quantity=0`在执行前失败。